# PCPP

In [4]:
import numpy as np

def sample_pcpp(n, theta, omega, random_state=None):
    if random_state is not None:
        np.random.seed(random_state)
    
    # 1. Número de comunidades potenciales
    N = np.random.poisson(theta)
    M = 1 + N
    
    # 2. Torneo aleatorio
    D = np.zeros(M, dtype=int)
    if M > 1:
        for j in range(M):
            for l in range(j + 1, M):
                win = np.random.binomial(1, 0.5)
                if win == 1:
                    D[j] += 1
                else:
                    D[l] += 1
                    
    # 3. Pesos de asignación
    W = ((1.0 - omega) / M) + (omega * (1.0 + 2.0 * D) / (M**2))
    
    # 4. Asignación de observaciones
    xi = np.random.choice(M, size=n, p=W)
    
    # 5. Formación de la partición (bloques no vacíos)
    unique_labels, counts = np.unique(xi, return_counts=True)
    K_n = len(unique_labels)
    
    # Mapeo a etiquetas canónicas {0, 1, ..., K_n - 1}
    label_map = {old_lbl: new_lbl for new_lbl, old_lbl in enumerate(unique_labels)}
    canonical_xi = np.array([label_map[lbl] for lbl in xi])
    
    # Representación en lista de bloques (subconjuntos de observaciones)
    blocks = [np.where(canonical_xi == k)[0].tolist() for k in range(K_n)]
    
    return {
        "M": M,
        "D": D,
        "W": W,
        "xi_raw": xi,
        "canonical_xi": canonical_xi,
        "K_n": K_n,
        "block_sizes": counts.tolist(),
        "blocks": blocks
    }

# Prueba de ejecución
res = sample_pcpp(n=10, theta=2.0, omega=0.7, random_state=42)
print("Resultados de prueba:")
for k, v in res.items():
    print(f"{k}: {v}")

Resultados de prueba:
M: 5
D: [2 3 2 1 2]
W: [0.2   0.256 0.2   0.144 0.2  ]
xi_raw: [0 1 2 1 1 2 0 1 1 2]
canonical_xi: [0 1 2 1 1 2 0 1 1 2]
K_n: 3
block_sizes: [2, 5, 3]
blocks: [[0, 6], [1, 3, 4, 7, 8], [2, 5, 9]]
